# End-to-End Insurance Retention Optimisation: ML + MIP

**Hands-on Jupyter notebook** | Synthetic motor insurance data | PuLP 3.x / 4.x

We start with simulated historical data from a **randomised offer pilot**, train renewal and conditional claims-cost models, score customer-offer scenarios, formulate a MIP, and compare the proposed portfolio with a no-discount baseline.

> **Training template, not production pricing guidance.** Real use requires causal validation, FCA-compliant offer eligibility and pricing, calibration, business sign-off, and monitored experiments. All results are estimated, not realised profit.

Run cells **top to bottom**. Change the parameters in the configuration cell and rerun downstream cells to see how strategy changes.

## 0. Dependencies
Run this cell only if packages are missing; restart the kernel if needed.

In [ ]:
# %pip install -U numpy pandas scikit-learn "pulp[cbc]" matplotlib

## 1. Configuration
All money figures are **GBP**. We model a maximum expected retention spend, a limited number of contacts, and an expected claims-exposure guardrail.

In [ ]:
SEED = 42
RNG = np.random.default_rng(SEED)
OUT = Path('insurance_retention_outputs')
OUT.mkdir(exist_ok=True)
OFFERS = (0, 5, 10)  # offer percentage
CONTACT_COST = 3.0
SERVICING_COST = 35.0
BUDGET = 18000.0
MAX_CONTACTS = 280
RISK_TOLERANCE = 1.06
N_HISTORICAL = 18000
N_TRAIN = 14000
N_FUTURE = 800

## 2. Generate historical insurance data

The simulator creates customer features, randomised historical discounts, renewal outcomes, and claims cost observed for renewers. Randomisation is crucial to the illustrative interpretation of offer response.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

In [ ]:
def build_customers(n, prefix):
    tenure = RNG.integers(1, 13, n)
    prior_claims = RNG.poisson(0.25, n).clip(0, 3)
    shopping = RNG.beta(2.2, 2.5, n)
    risk = RNG.beta(2.4, 4.0, n)
    premium = np.round(470 + 560 * risk + 35 * prior_claims + RNG.normal(0, 90, n), 2).clip(300, 1200)
    return pd.DataFrame({
        'customer_id': [f'{prefix}{i:06}' for i in range(n)],
        'tenure_years': tenure, 'prior_claims': prior_claims,
        'shopping_score': shopping, 'risk_score': risk,
        'premium': premium,
    })

In [ ]:
def true_renewal_prob(df):
    discount = df['offer_pct'].to_numpy() / 100
    return sigmoid(
        0.55 + 0.065 * df['tenure_years'].to_numpy()
        - 1.9 * df['shopping_score'].to_numpy()
        - 0.35 * df['risk_score'].to_numpy()
        - 0.20 * df['prior_claims'].to_numpy()
        + discount * (2.0 + 9.0 * df['shopping_score'].to_numpy()
                      - 0.08 * df['tenure_years'].to_numpy())
    )

In [ ]:
def true_conditional_claim_cost(df):
    return (180 + 520 * df['risk_score'].to_numpy()
            + 80 * df['prior_claims'].to_numpy())

In [ ]:
def simulate_pilot(n, prefix):
    df = build_customers(n, prefix)
    # IMPORTANT: Randomised historic pilot: allows identification of offer effects.
    df['offer_pct'] = RNG.choice(OFFERS, n, p=[0.5, 0.3, 0.2])
    prob = true_renewal_prob(df)
    df['renewed'] = RNG.binomial(1, prob)
    claim_mean = true_conditional_claim_cost(df)
    # Observed claim cost is only available if the policy renewed.
    df['claim_cost_if_renewed'] = np.where(
        df['renewed'].eq(1), RNG.gamma(shape=8, scale=claim_mean / 8), np.nan
    )
    return df

In [ ]:
pilot = simulate_pilot(N_HISTORICAL, 'H')
train = pilot.iloc[:N_TRAIN].copy()
valid = pilot.iloc[N_TRAIN:].copy()
print('Historical:', pilot.shape, '| Train:', train.shape, '| Valid:', valid.shape)
display(pilot.head())
print('Renewal rate by randomised offer:')
display(pilot.groupby('offer_pct')['renewed'].agg(['count', 'mean']))

## 3. Feature engineering and predictive models

We fit logistic regression for renewal and Ridge regression for **claims cost conditional on renewal**. Validate predictions on a held-out sample.

In [ ]:
def features(df):
    """The SAME transformations must be used during model training and scoring."""
    offer = df['offer_pct'].to_numpy() / 100
    return pd.DataFrame({
        'tenure': df['tenure_years'].to_numpy(),
        'prior_claims': df['prior_claims'].to_numpy(),
        'shopping': df['shopping_score'].to_numpy(),
        'risk': df['risk_score'].to_numpy(),
        'premium': df['premium'].to_numpy(),
        'offer': offer,
        'offer_x_shopping': offer * df['shopping_score'].to_numpy(),
        'offer_x_tenure': offer * df['tenure_years'].to_numpy(),
    })

In [ ]:
def train_and_evaluate(train, valid):
    response_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500))
    response_model.fit(features(train), train['renewed'])
    pred = response_model.predict_proba(features(valid))[:, 1]
    print(f'Renewal validation: AUC={roc_auc_score(valid.renewed, pred):.3f}, '
          f'Brier={brier_score_loss(valid.renewed, pred):.3f}')
    # This is a model of claim cost CONDITIONAL ON RENEWAL.
    claim_train = train.loc[train['renewed'].eq(1)].copy()
    claim_valid = valid.loc[valid['renewed'].eq(1)].copy()
    claim_columns = ['risk_score', 'prior_claims']
    claim_model = make_pipeline(StandardScaler(), Ridge(alpha=20))
    claim_model.fit(claim_train[claim_columns], claim_train['claim_cost_if_renewed'])
    valid_cost = claim_model.predict(claim_valid[claim_columns]).clip(0)
    print(f'Conditional claims validation MAE=£'
          f'{mean_absolute_error(claim_valid.claim_cost_if_renewed, valid_cost):.2f}')
    return response_model, claim_model

In [ ]:
response_model, claim_model = train_and_evaluate(train, valid)

## 4. Construct customer–offer scenarios

Score each future customer under all offers; compute net expected profit, expected incentive spend, loss exposure and offer eligibility.

In [ ]:
def offer_eligibility(row, offer):
    # Illustrative approved operational policy rules; not an assertion of regulatory compliance.
    if offer == 10 and row.tenure_years < 2:
        return False
    if offer == 10 and row.risk_score > 0.65:
        return False
    return True

In [ ]:
def score_scenarios(future, response_model, claim_model):
    candidates = future.loc[future.index.repeat(len(OFFERS))].copy().reset_index(drop=True)
    candidates['offer_pct'] = np.tile(OFFERS, len(future))
    # Explicitly score the counterfactual scenario for each eligible offer.
    candidates['p_renew'] = response_model.predict_proba(features(candidates))[:, 1]
    candidates['expected_claim_if_renewed'] = claim_model.predict(
        candidates[['risk_score', 'prior_claims']]
    ).clip(0)
    discount = candidates['offer_pct'] / 100
    candidates['discount_amount'] = candidates['premium'] * discount
    candidates['contact_cost'] = np.where(candidates.offer_pct.gt(0), CONTACT_COST, 0.0)
    # Premium already includes offer reduction; never subtract discount twice.
    candidates['expected_profit'] = (
        candidates['p_renew'] * (
            candidates['premium'] * (1 - discount)
            - candidates['expected_claim_if_renewed'] - SERVICING_COST
        ) - candidates['contact_cost']
    )
    # Expected redeemed incentive, plus contact cost. This is the budget convention.
    candidates['budget_cost'] = (
        candidates['p_renew'] * candidates['discount_amount'] + candidates['contact_cost']
    )
    candidates['expected_loss_exposure'] = (
        candidates['p_renew'] * candidates['expected_claim_if_renewed']
    )
    allowed = [offer_eligibility(row, int(row.offer_pct))
               for row in candidates.itertuples(index=False)]
    candidates['eligible'] = allowed
    assert candidates.groupby('customer_id').apply(
        lambda g: bool(g.loc[g.offer_pct.eq(0), 'eligible'].all()),
        include_groups=False
    ).all()
    return candidates

In [ ]:
future = build_customers(N_FUTURE, 'F')
candidates = score_scenarios(future, response_model, claim_model)
print('Scored scenarios:', candidates.shape)
display(candidates[['customer_id','offer_pct','p_renew','expected_claim_if_renewed','expected_profit','budget_cost','eligible']].head(9))
print('Eligible offers:', candidates.groupby('offer_pct')['eligible'].sum().to_dict())

## 5. MIP allocation

Binary decision variables are **flat dictionaries with tuple keys**: `x[(customer_id, offer_pct)]`. The objective maximises already-net expected profit; do **not** subtract the discount again.

In [ ]:
def make_solver():
    if hasattr(pulp, 'COIN_CMD'):
        try:
            solver = pulp.COIN_CMD(msg=False, timeLimit=90)
            if solver.available():
                return solver
        except Exception:
            pass
    if hasattr(pulp, 'PULP_CBC_CMD'):
        solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=90)
        if solver.available():
            return solver
    raise RuntimeError('CBC not found. For PuLP 4 install with: pip install "pulp[cbc]"')

In [ ]:
def solve_portfolio(candidates):
    eligible = candidates.loc[candidates.eligible].copy()
    pairs = list(zip(eligible.customer_id, eligible.offer_pct.astype(int)))
    indexed = eligible.set_index(['customer_id', 'offer_pct'])
    customers = candidates.customer_id.unique().tolist()
    model = pulp.LpProblem('Renewal_offer_allocation', pulp.LpMaximize)
    if hasattr(model, 'add_variable_dict'):
        x = model.add_variable_dict('choose', pairs, cat='Binary', lowBound=0, upBound=1)
    else:
        x = pulp.LpVariable.dicts('choose', pairs, lowBound=0, upBound=1, cat='Binary')
    assert all(x[k].isBinary() for k in pairs), 'Binary variable creation failed'

    profits = indexed.expected_profit.to_dict()
    spend = indexed.budget_cost.to_dict()
    exposures = indexed.expected_loss_exposure.to_dict()
    total_profit = pulp.lpSum(profits[k] * x[k] for k in pairs)
    total_spend = pulp.lpSum(spend[k] * x[k] for k in pairs)
    total_exposure = pulp.lpSum(exposures[k] * x[k] for k in pairs)
    model += total_profit, 'Expected_net_profit'
    for customer in customers:
        cust_pairs = [k for k in pairs if k[0] == customer]
        model += pulp.lpSum(x[k] for k in cust_pairs) == 1, f'One_offer_{customer}'
    model += total_spend <= BUDGET, 'Retention_budget'
    model += pulp.lpSum(x[k] for k in pairs if k[1] > 0) <= MAX_CONTACTS, 'Contact_capacity'
    baseline_exposure = float(candidates.loc[candidates.offer_pct.eq(0), 'expected_loss_exposure'].sum())
    model += total_exposure <= RISK_TOLERANCE * baseline_exposure, 'Loss_exposure_guardrail'
    result = model.solve(make_solver())
    status = getattr(result, 'status_str', None)
    if status is None:
        status = pulp.LpStatus[model.status]
    if status != 'Optimal':
        raise RuntimeError(f'No proven optimal solution (status={status}). Inspect model/solver log.')
    selected_pairs = [k for k in pairs if pulp.value(x[k]) > 0.5]
    assert len(selected_pairs) == len(customers)
    decisions = eligible.set_index(['customer_id', 'offer_pct']).loc[selected_pairs].reset_index()
    assert decisions.customer_id.is_unique
    assert float(decisions.budget_cost.sum()) <= BUDGET + 1e-4
    assert int(decisions.offer_pct.gt(0).sum()) <= MAX_CONTACTS
    assert float(decisions.expected_loss_exposure.sum()) <= RISK_TOLERANCE * baseline_exposure + 1e-4
    assert abs(float(decisions.expected_profit.sum()) - float(pulp.value(total_profit))) < 1e-4
    print('Solver:', status, '| Variables:', len(pairs), '| Constraints:', len(model.constraints()) if callable(model.constraints) else len(model.constraints))
    return decisions

In [ ]:
decisions = solve_portfolio(candidates)
display(decisions[['customer_id','offer_pct','p_renew','expected_profit','budget_cost']].head(10))

## 6. Evaluate and export

Compare the selected offer policy with no discounts. These are predicted outcomes, not observed treatment effects.

In [ ]:
def metrics(df):
    return {
        'expected_profit': round(float(df.expected_profit.sum()), 2),
        'expected_budget_spend': round(float(df.budget_cost.sum()), 2),
        'expected_renewals': round(float(df.p_renew.sum()), 2),
        'expected_loss_exposure': round(float(df.expected_loss_exposure.sum()), 2),
        'customers_contacted': int(df.offer_pct.gt(0).sum()),
    }

In [ ]:
baseline = candidates.loc[candidates.offer_pct.eq(0)].copy()
summary = pd.DataFrame({'No discount':metrics(baseline), 'Optimised':metrics(decisions)})
summary['Change'] = summary['Optimised'] - summary['No discount']
display(summary)
print('Offer distribution:')
display(decisions.offer_pct.value_counts().sort_index().rename('customers').to_frame())

In [ ]:
candidates.to_csv(OUT / 'scored_customer_offer_scenarios.csv', index=False)
decisions.to_csv(OUT / 'optimised_customer_offers.csv', index=False)
summary.to_csv(OUT / 'strategy_comparison.csv')
print('Saved outputs to:', OUT.resolve())

## 7. Interpretation and experiments

Try changing `BUDGET`, `MAX_CONTACTS`, and `RISK_TOLERANCE` in the configuration cell; rebuild the MIP by rerunning the **solve** and **evaluation** cells (re-score if offer eligibility/model inputs change). Ask:

- How much additional expected profit does the optimiser generate over no offer?
- How many customers receive each offer, and what is total expected spend?
- Are the budget, capacity or risk constraints binding?
- What happens if customer probabilities are miscalibrated or offer effects are less than predicted?

**Production checklist:** secure historical feature snapshots, remove outcome leakage, validate randomisation or identify treatment effects causally, estimate uncertainty, comply with FCA pricing / fairness rules, check capital and loss assumptions, log decisions, design holdout/A/B testing and monitor realised outcomes.